<a href="https://colab.research.google.com/github/diegopaucarv/nlp-ai-tools/blob/main/pipeline_radio_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📻 Pipeline de Análisis de Radio: YAMNet + FunASR + Faster-Whisper

Este notebook está diseñado para procesar archivos largos de audio (ej. 30 minutos). Combina tres herramientas:
1. **YAMNet**: Detecta eventos acústicos (Música, Anuncios/Jingles, Silencios, Efectos) y devuelve marcas de tiempo.
2. **FunASR (SenseVoiceSmall)**: Transcribe el texto incluyendo etiquetas semánticas de audio (ej. `<|BGM|>` para música de fondo).
3. **Faster-Whisper**: Proporciona una transcripción altamente veloz con marcas de tiempo (timestamps) exactas para cada segmento de voz.

**⚠️ IMPORTANTE ANTES DE EMPEZAR:**
Ve al menú superior: `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> Selecciona **A100**. Esto acelerará el procesamiento de 30 minutos a solo un par de minutos.

In [ ]:
#
# ===============================================================
# CELDA 1: ENTORNO UNIFICADO SOTA (RESOLUCIÓN FORZADA DE PIP)
# ===============================================================
print("⏳ Resolviendo el Dependency Hell (Forzando compatibilidad en Colab)...")

# 1. ACTUALIZAR PIP PRIMERO
# Un pip viejo no sabe resolver conflictos de forma inteligente
!python -m pip install --upgrade pip

# 2. INSTALAR LIBRERÍAS DE SISTEMA
!apt-get install -y -qq ffmpeg

# 3. INSTALAR LAS PIEZAS BASE INAMOVIBLES (Fijando versiones de conflicto)
# Obligamos a instalar un omegaconf moderno que WhisperX acepte
!pip install -q "numpy==2.2.3" "pandas==2.2.3" "scipy>=1.14.0" "omegaconf>=2.3.0" "hydra-core>=1.3.2"

# 4. INSTALAR EL STACK DE AUDIO PRINCIPAL
!pip install -q pyannote.audio demucs librosa transformers soundfile ctranslate2 speechbrain

# 5. INSTALAR FASTER-WHISPER CON CUIDADO
# Lo instalamos ignorando las dependencias que ya pusimos en el paso 3
!pip install -q --no-deps faster-whisper
# Añadimos solo las dependencias seguras que necesita faster-whisper
!pip install -q av onnxruntime tokenizers

# 6. INSTALAR DENOISER A LA FUERZA
# Denoiser exige un hydra_core y omegaconf prehistóricos. Se lo prohibimos.
!pip install -q --no-deps denoiser noisereduce
!pip install -q pystoi sounddevice julius

# 7. INSTALAR WHISPERX (CON PRECAUCIÓN MÁXIMA)
# WhisperX también exige versiones muy específicas de cosas que romperían Colab
!pip install -q --no-deps git+https://github.com/m-bain/whisperx.git
!pip install -q nltk

# 8. PARCHE DE EMERGENCIA PARA COLAB
# Nos aseguramos de que el ecosistema de PyTorch original de Colab no haya sido tocado
!pip install -q "torch==2.11.0+cu130" "torchaudio==2.11.0+cu130" "torchvision==0.26.0+cu130" --extra-index-url https://download.pytorch.org/whl/cu130

!pip install ruptures nara_wpe

print("✅ Entorno listo. Reiniciando sesión para limpiar memoria...")
import os
os.kill(os.getpid(), 9)

⏳ Resolviendo el Dependency Hell (Forzando compatibilidad en Colab)...
ERROR: Blocked import of 'importlib.metadata._adapters' after pip install started. Hook authors must import required modules before installation starts.
ERROR: Blocked import of 'importlib.metadata._adapters' after pip install started. Hook authors must import required modules before installation starts.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
# ===============================================================
# PARCHE DEFINITIVO: INYECCIÓN DE LIBRERÍAS CUDA 12 EN EL SISTEMA
# ===============================================================
import os
import glob
import site

print("🛠️ Instalando librerías puente de CUDA 12...")
os.system("pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12")

print("🔗 Inyectando librerías dinámicas en /usr/lib/...")
site_packages = site.getsitepackages()[0]

rutas_origen = [
    os.path.join(site_packages, "nvidia", "cublas", "lib", "*.so*"),
    os.path.join(site_packages, "nvidia", "cudnn", "lib", "*.so*")
]

contador = 0
for ruta in rutas_origen:
    for archivo in glob.glob(ruta):
        nombre = os.path.basename(archivo)
        destino = f"/usr/lib/{nombre}"
        # Forzar el enlace simbólico en la carpeta raíz de librerías de Linux
        os.system(f"ln -sf {archivo} {destino}")
        contador += 1

# Refrescar la caché del enlazador de Linux
os.system("ldconfig")

print(f"✅ ¡Éxito! Se vincularon {contador} librerías de NVIDIA directamente al sistema.")
print("🚨 IMPORTANTE: Ahora debes reiniciar la sesión.")

🛠️ Instalando librerías puente de CUDA 12...
🔗 Inyectando librerías dinámicas en /usr/lib/...
✅ ¡Éxito! Se vincularon 11 librerías de NVIDIA directamente al sistema.
🚨 IMPORTANTE: Ahora debes reiniciar la sesión.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# audio_pipeline_v5.py
# - custom emotion head (audEERING regression head)
# - scipy WPE (no nara_wpe)
# - direct diarization speaker (no word-vote)
# - short ASR initial_prompt
# =========================================================
import os, sys, gc, json, math, shutil, logging, time, subprocess, warnings
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import soundfile as sf
import librosa
import psutil
from scipy.signal import stft as scipy_stft, istft as scipy_istft

warnings.filterwarnings("ignore")
SR = 16000

# ---------------------------------------------------------
# LOGGING + MEMORY TELEMETRY
# ---------------------------------------------------------
def setup_logger(log_dir: str) -> logging.Logger:
    os.makedirs(log_dir, exist_ok=True)
    log = logging.getLogger("v5")
    log.setLevel(logging.DEBUG)
    log.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
    for h in (logging.StreamHandler(sys.stdout),
              logging.FileHandler(os.path.join(log_dir, "pipeline_v5.log"), mode="w")):
        h.setFormatter(fmt); log.addHandler(h)
    return log

LOG = setup_logger("/content/drive/MyDrive/resultados_radio/_logs")
PROC = psutil.Process()

def mem(tag: str):
    ram = PROC.memory_info().rss / 1024 / 1024
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1024 / 1024
        r = torch.cuda.memory_reserved() / 1024 / 1024
        LOG.info(f"[MEM {tag:<24}] RAM {ram:7.0f} MB | GPU alloc {a:6.0f} MB | rsvd {r:6.0f} MB")
    else:
        LOG.info(f"[MEM {tag:<24}] RAM {ram:7.0f} MB")

def clear():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------------------------------------
# STAGE 0 — normalize (streaming, ffmpeg only)
# ---------------------------------------------------------
def normalize_ffmpeg(src: str, dst: str, sr: int = SR) -> str:
    subprocess.run(
        ["ffmpeg", "-y", "-i", src, "-vn",
         "-ac", "1", "-ar", str(sr), "-c:a", "pcm_s16le", dst],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return dst

# ---------------------------------------------------------
# DENOISER BACKENDS
# ---------------------------------------------------------
class FacebookDenoiserWrapper:
    """facebookresearch/denoiser (dns48 / dns64 / master64). 16 kHz native."""
    def __init__(self, model_name: str = "dns48", device: Optional[str] = None,
                 wet: float = 1.0):
        from denoiser import pretrained
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = model_name
        self.wet = float(wet)
        if model_name == "dns48":
            self.model = pretrained.dns48()
        elif model_name == "dns64":
            self.model = pretrained.dns64()
        elif model_name == "master64":
            self.model = pretrained.master64()
        else:
            raise ValueError(f"Unknown FB denoiser: {model_name}")
        self.model = self.model.to(self.device).eval()
        self.sample_rate = int(self.model.sample_rate)
        LOG.info(f"FacebookDenoiser ready | model={model_name} sr={self.sample_rate} "
                 f"device={self.device} wet={self.wet}")

    @torch.inference_mode()
    def process(self, y: np.ndarray) -> np.ndarray:
        t = torch.from_numpy(y).float().unsqueeze(0).unsqueeze(0).to(self.device)
        out = self.model(t).squeeze().cpu().numpy().astype(np.float32)
        if len(out) > len(y): out = out[:len(y)]
        elif len(out) < len(y): out = np.pad(out, (0, len(y) - len(out)))
        if self.wet < 1.0:
            out = self.wet * out + (1.0 - self.wet) * y
        return out


class NoisereduceWrapper:
    """Spectral gating. Fast, no GPU. prop_decrease 0.7–0.85 for interviews."""
    def __init__(self, prop_decrease: float = 0.8, n_fft: int = 1024):
        import noisereduce as nr
        self.nr = nr
        self.prop_decrease = float(prop_decrease)
        self.n_fft = int(n_fft)
        self.sample_rate = SR
        LOG.info(f"Noisereduce ready | prop_decrease={prop_decrease} n_fft={n_fft}")

    def process(self, y: np.ndarray) -> np.ndarray:
        z = self.nr.reduce_noise(
            y=y, sr=SR, stationary=False,
            prop_decrease=self.prop_decrease,
            n_fft=self.n_fft,
            time_constant_s=0.5,
            freq_mask_smooth_hz=500,
            time_mask_smooth_ms=50,
        )
        return z.astype(np.float32)


def build_denoiser(name: str, device: Optional[str] = None,
                   fb_model: str = "dns48", fb_wet: float = 1.0,
                   nr_prop: float = 0.8):
    if name in ("none", "afftdn"):
        return None
    if name == "facebook":
        return FacebookDenoiserWrapper(model_name=fb_model, device=device, wet=fb_wet)
    if name == "noisereduce":
        return NoisereduceWrapper(prop_decrease=nr_prop)
    raise ValueError(f"Unknown denoiser: {name}")


def denoise_file(src: str, dst: str, backend, name: str,
                 chunk_s: int, xfade_s: float = 0.5) -> str:
    if name == "afftdn":
        af = "highpass=f=70,lowpass=f=7800,afftdn=nr=12:nf=-25"
        subprocess.run(["ffmpeg", "-y", "-i", src, "-af", af,
                        "-ar", str(SR), "-ac", "1", "-c:a", "pcm_s16le", dst],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return dst

    if backend is None:
        shutil.copy(src, dst); return dst

    info = sf.info(src)
    total = info.frames
    cf = chunk_s * backend.sample_rate
    xf = int(xfade_s * backend.sample_rate)
    n_chunks = max(1, math.ceil(total / cf))

    prev_tail = None
    pos = 0
    i = 0
    t0 = time.time()
    with sf.SoundFile(dst, "w", samplerate=backend.sample_rate,
                      channels=1, subtype="PCM_16") as fout:
        while pos < total:
            i += 1
            frames = min(cf, total - pos)
            y, _ = sf.read(src, start=pos, frames=frames, dtype="float32")
            if y.ndim > 1: y = y.mean(1)
            z = backend.process(y)
            if prev_tail is not None and len(z) > xf:
                w = np.linspace(0, 1, xf, dtype=np.float32)
                z[:xf] = prev_tail[:xf] * (1 - w) + z[:xf] * w
            fout.write(z)
            prev_tail = z[-xf:].copy() if len(z) > xf else None
            del y, z
            rtf = (pos + frames) / backend.sample_rate / max(time.time() - t0, 1e-6)
            LOG.info(f"      denoise {i}/{n_chunks} | RTF {rtf:.1f}x")
            mem(f"denoise chunk {i}")
            pos += cf
    return dst

# ---------------------------------------------------------
# REVERB SCORE (sampled)
# ---------------------------------------------------------
def _reverb_score_window(y: np.ndarray, sr: int = SR) -> float:
    if len(y) < sr: return 0.0
    hop = 256
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    from scipy.signal import find_peaks
    peaks, _ = find_peaks(rms, height=np.percentile(rms, 95), distance=10)
    if len(peaks) < 3: return 0.0
    tail = int(0.20 * sr / hop)
    ratios, slopes = [], []
    for p in peaks:
        if p + tail >= len(rms): continue
        pk = max(rms[p], 1e-9)
        ratios.append((np.mean(rms[p + tail // 2: p + tail]) + 1e-9) / pk)
        seg = rms[p:p + tail] + 1e-9
        slopes.append(-np.polyfit(np.arange(len(seg)), np.log(seg), 1)[0])
    if not ratios: return 0.0
    return float(np.median(ratios) * 5.0 + np.median(slopes) * 0.5)

def reverb_score_sampled(path: str, n_windows: int = 5, window_s: int = 10) -> float:
    info = sf.info(path); total = info.frames; wf = window_s * SR
    if total <= wf:
        y, _ = sf.read(path, dtype="float32")
        if y.ndim > 1: y = y.mean(1)
        return _reverb_score_window(y)
    scores = []
    for i in range(n_windows):
        start = int(i * (total - wf) / max(n_windows - 1, 1))
        y, _ = sf.read(path, start=start, frames=wf, dtype="float32")
        if y.ndim > 1: y = y.mean(1)
        s = _reverb_score_window(y)
        scores.append(s)
        del y
    clear()
    return float(np.median(scores)) if scores else 0.0

# ---------------------------------------------------------
# WPE — pure scipy/numpy, no native extensions
# ---------------------------------------------------------
def _wpe_window(y: np.ndarray,
                taps: int = 10, delay: int = 3, iterations: int = 3,
                n_fft: int = 512, hop: int = 128, reg: float = 1e-3) -> np.ndarray:
    if y.dtype != np.float32:
        y = y.astype(np.float32)

    _, _, Z = scipy_stft(y, fs=SR, nperseg=n_fft, noverlap=n_fft - hop,
                         boundary="zeros", padded=True)
    Y = np.ascontiguousarray(Z.T, dtype=np.complex64)      # [T, F] — never modified
    T, F = Y.shape
    if T < delay + taps + 2:
        return y.copy()

    # Xd[k, f, t] = Y[t - (delay + k), f]
    Xd = np.zeros((taps, F, T), dtype=np.complex64)
    for k in range(taps):
        sh = delay + k
        if sh < T:
            Xd[k, :, sh:] = Y[:-sh, :].T
    Xd_conj = np.conj(Xd)
    Yc = np.conj(Y)                                        # [T, F]

    X = Y.copy()
    eye = np.eye(taps, dtype=np.complex64)[:, :, None]
    for _ in range(iterations):
        # power estimate from CURRENT output, floored relative to mean
        p = (X.real**2 + X.imag**2)                        # [T, F]
        p = np.maximum(p, 1e-4 * p.mean() + 1e-10)
        inv = (1.0 / p).T.astype(np.float32)               # [F, T]

        Xw = Xd * inv[None]                                # [K, F, T]
        R = np.einsum("kft,lft->klf", Xw, Xd_conj, optimize=True)   # [K,K,F]
        P = np.einsum("kft,tf->kf",  Xw, Yc,      optimize=True)    # [K,F]

        tr = np.trace(R, axis1=0, axis2=1).real
        R += (reg * np.maximum(tr / taps, 1e-9))[None, None, :] * eye

        G = np.linalg.solve(np.moveaxis(R, 2, 0), P.T[..., None])[..., 0]  # [F,K]

        # X = Y - conj(G)^T · Xd   (always from the original Y)
        X = Y - np.einsum("fk,kft->tf", np.conj(G), Xd, optimize=True)

    _, y_out = scipy_istft(X.T, fs=SR, nperseg=n_fft, noverlap=n_fft - hop)
    y_out = y_out.astype(np.float32)
    if len(y_out) > len(y): y_out = y_out[:len(y)]
    elif len(y_out) < len(y): y_out = np.pad(y_out, (0, len(y) - len(y_out)))
    return y_out

def wpe_file_chunked(src: str, dst: str,
                     chunk_s: int = 15, pad_s: int = 1,
                     xfade_s: float = 0.3,
                     taps: int = 10, delay: int = 3, iterations: int = 3,
                     mem_guard_pct: float = 85.0) -> Optional[str]:
    info = sf.info(src); total = info.frames
    cf, pf = chunk_s * SR, pad_s * SR
    xf = int(xfade_s * SR)
    n_chunks = max(1, math.ceil(total / cf))

    prev_tail = None
    pos = 0
    i = 0
    t0 = time.time()
    with sf.SoundFile(dst, "w", samplerate=SR, channels=1, subtype="PCM_16") as fout:
        while pos < total:
            i += 1
            vm = psutil.virtual_memory().percent
            if vm > mem_guard_pct:
                clear()
                vm = psutil.virtual_memory().percent
                if vm > mem_guard_pct + 5:
                    LOG.error(f"[WPE] RAM at {vm:.0f}% — aborting, returning pre-WPE audio")
                    return None

            start = max(0, pos - pf)
            end = min(total, pos + cf + pf)
            y, _ = sf.read(src, start=start, frames=end - start, dtype="float32")
            if y.ndim > 1: y = y.mean(1)

            z = _wpe_window(y, taps=taps, delay=delay, iterations=iterations)

            cs = pos - start
            cl = min(cf, total - pos)
            core = z[cs:cs + cl].copy()

            if prev_tail is not None and len(core) > xf:
                w = np.linspace(0, 1, xf, dtype=np.float32)
                core[:xf] = prev_tail[:xf] * (1 - w) + core[:xf] * w

            fout.write(core)
            prev_tail = core[-xf:].copy() if len(core) > xf else None

            del y, z, core
            gc.collect()

            elapsed = time.time() - t0
            rtf = (pos + cl) / SR / max(elapsed, 1e-6)
            LOG.info(f"      WPE {i}/{n_chunks} ({(pos+cl)/SR:.0f}s/{total/SR:.0f}s) "
                     f"| RTF {rtf:.1f}x")
            mem(f"WPE chunk {i}")
            pos += cf
    clear()
    return dst

# ---------------------------------------------------------
# EMOTION — custom audEERING regression head (fixes MISSING weights)
# ---------------------------------------------------------
class _RegressionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden = config.hidden_size
        dropout = getattr(config, "final_dropout", None) or \
                  getattr(config, "classifier_dropout", 0.1)
        self.dense = nn.Linear(hidden, hidden)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(hidden, config.num_labels)

    def forward(self, features):
        x = self.dropout(features)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        return self.out_proj(x)


def _build_emotion_model_classes():
    """Late import so this file can be loaded without transformers on path."""
    from transformers import Wav2Vec2PreTrainedModel, Wav2Vec2Model, Wav2Vec2Config

    class EmotionModel(Wav2Vec2PreTrainedModel):
        # These satisfy newer transformers' loading path. Our head has no
        # tied weights, so both are empty.
        _tied_weights_keys = []
        all_tied_weights_keys = {}

        def __init__(self, config: Wav2Vec2Config):
            super().__init__(config)
            self.wav2vec2 = Wav2Vec2Model(config)
            self.classifier = _RegressionHead(config)
            self.init_weights()

        def forward(self, input_values):
            outputs = self.wav2vec2(input_values)
            hidden = outputs[0].mean(dim=1)
            logits = self.classifier(hidden)
            return hidden, logits

    return EmotionModel

def load_emotion(model_id: str, device: str):
    from transformers import Wav2Vec2Processor, Wav2Vec2Config
    EmotionModel = _build_emotion_model_classes()

    proc = Wav2Vec2Processor.from_pretrained(model_id)
    config = Wav2Vec2Config.from_pretrained(model_id)
    config.num_labels = 3

    model = EmotionModel.from_pretrained(model_id, config=config)
    model = model.to(device).eval()

    # Sanity check — this is the whole point of the rewrite
    dense_ok = hasattr(model.classifier, "dense") and hasattr(model.classifier, "out_proj")
    if dense_ok:
        w = model.classifier.out_proj.weight
        LOG.info(f"Emotion head OK = True | out_proj {tuple(w.shape)} "
                 f"| abs.mean={w.abs().mean().item():.4f}")
    else:
        LOG.error("Emotion head missing — regression head did not load")
    return proc, model


@torch.inference_mode()
def emotion_streaming(path: str, segments: List[dict], proc, model) -> List[dict]:
    out = []
    dev = model.device
    for k, seg in enumerate(segments):
        s = int(seg["start"] * SR); e = int(seg["end"] * SR); n = e - s
        if n < int(0.1 * SR):
            continue
        try:
            y, _ = sf.read(path, start=s, frames=n, dtype="float32")
        except Exception:
            continue
        if y.ndim > 1: y = y.mean(1)

        if len(y) <= 3 * SR:
            inp = proc(y, sampling_rate=SR, return_tensors="pt").to(dev)
            _, logits = model(inp["input_values"])
            a, d, v = logits[0].tolist()
            del inp, logits
        else:
            aL, dL, vL = [], [], []
            for t0 in np.arange(0, len(y)/SR - 3.0 + 1e-3, 1.0):
                sub = y[int(t0*SR):int((t0+3.0)*SR)]
                inp = proc(sub, sampling_rate=SR, return_tensors="pt").to(dev)
                _, logits = model(inp["input_values"])
                aL.append(logits[0, 0].item())
                dL.append(logits[0, 1].item())
                vL.append(logits[0, 2].item())
                del inp, logits
            a, d, v = float(np.mean(aL)), float(np.mean(dL)), float(np.mean(vL))

        out.append({**seg, "arousal": a, "dominance": d, "valence": v})
        if (k + 1) % 50 == 0:
            clear(); mem(f"emotion seg {k+1}/{len(segments)}")
        del y
    return out


def label3(v, m, s):
    if v > m + 0.5*s: return "ALTA"
    if v < m - 0.5*s: return "BAJA"
    return "MEDIA"

# ---------------------------------------------------------
# ASR — short prompt, no word-vote
# ---------------------------------------------------------
def transcribe(path, whisper, align_model, align_meta, diarize,
               min_spk, max_spk):
    import whisperx
    audio = whisperx.load_audio(path)
    mem("after load_audio")

    result = whisper.transcribe(
        audio,
        batch_size=8,
        language="es",
        task="transcribe",
    )
    del audio; clear(); mem("after transcribe")

    result = whisperx.align(
        result["segments"], align_model, align_meta,
        whisperx.load_audio(path), "cuda",
        return_char_alignments=False,
    )
    clear(); mem("after align")

    diar = diarize(path, min_speakers=min_spk, max_speakers=max_spk)
    clear(); mem("after diarize")

    from whisperx.diarize import assign_word_speakers
    result = assign_word_speakers(diar, result)

    out = []
    for s in result["segments"]:
        text = s["text"].strip()
        if not text:
            continue
        # Use the segment speaker directly — no word-vote.
        # `assign_word_speakers` already assigns a dominant speaker per segment.
        spk = s.get("speaker") or "UNKNOWN"
        out.append({"start": s["start"], "end": s["end"],
                    "text": text, "speaker": spk})
    return out

# ---------------------------------------------------------
# WRITERS
# ---------------------------------------------------------
def fmt_ts(s):
    h = int(s // 3600); m = int((s % 3600) // 60); sec = int(s % 60)
    return f"{h:02d}:{m:02d}:{sec:02d},{int((s - int(s))*1000):03d}"

def write_srt(path, segs):
    with open(path, "w", encoding="utf-8") as f:
        i = 1
        for s in segs:
            t = s["text"].strip()
            if not t: continue
            f.write(f"{i}\n{fmt_ts(s['start'])} --> {fmt_ts(s['end'])}\n"
                    f"{s['speaker']}: {t}\n\n")
            i += 1

def write_csv(path, segs):
    import csv
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["start","end","speaker","valence","arousal","dominance",
                    "valence_label","arousal_label","dominance_label","text"])
        for s in segs:
            w.writerow([s["start"], s["end"], s["speaker"],
                        s.get("valence"), s.get("arousal"), s.get("dominance"),
                        s.get("valence_label"), s.get("arousal_label"),
                        s.get("dominance_label"),
                        s["text"].replace("\n", " ")])

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
@dataclass
class Cfg:
    input_dir: str
    output_dir: str
    temp_dir: str
    log_dir: str
    hf_token: Optional[str] = None

    # toggles
    denoise_method: str = "facebook"
    do_wpe: bool = True
    do_emotion: bool = True
    do_diarization: bool = True
    emotion_on_clean_path: bool = True
    keep_temp: bool = False

    # reverb
    reverb_threshold: float = 0.30

    # denoiser
    denoiser_device: Optional[str] = None
    fb_model: str = "dns48"
    fb_wet: float = 1.0
    nr_prop: float = 0.8
    denoise_chunk_s: int = 30
    denoise_xfade_s: float = 0.5

    # WPE (scipy)
    wpe_chunk_s: int = 15
    wpe_pad_s: int = 1
    wpe_xfade_s: float = 0.3
    wpe_taps: int = 10
    wpe_delay: int = 3
    wpe_iters: int = 3
    mem_guard_pct: float = 85.0

    # ASR
    whisper_size: str = "large-v3"
    min_speakers: int = 1
    max_speakers: int = 6

    video_exts: Tuple[str, ...] = (".mp4", ".mkv", ".mov", ".mpeg", ".mpg")
    audio_exts: Tuple[str, ...] = (".mp3", ".wav", ".m4a", ".flac", ".ogg")


# ---------------------------------------------------------
# PIPELINE
# ---------------------------------------------------------
class Pipeline:
    def __init__(self, cfg: Cfg):
        self.cfg = cfg
        for d in (cfg.input_dir, cfg.output_dir, cfg.temp_dir, cfg.log_dir):
            os.makedirs(d, exist_ok=True)
        self.whisper = self.align = self.align_meta = self.diar = None
        self.proc = self.emotion = None
        self.denoiser = None

    def _dev(self): return "cuda" if torch.cuda.is_available() else "cpu"

    def _load_asr(self):
        if self.whisper is not None:
            return
        import whisperx
        from huggingface_hub import login
        if self.cfg.hf_token:
            login(token=self.cfg.hf_token, add_to_git_credential=False)
        dev = self._dev()
        LOG.info(f"Loading WhisperX {self.cfg.whisper_size} on {dev}")

        # Short prompt: nouns only. No verbs, no framing. This kills the
        # "Términos en español peruano" leak observed in the previous run.
        asr_options = {
            "initial_prompt": (
                "Uyarina Wasi. Ayna. San Francisco. INABIF. CEM. OPS. ENSAP. "
                "Quechua. Defensoría. Municipalidad. Centro de Salud Mental "
                "Comunitario. Trabajadora social. Intervención en crisis. "
                "Brigadistas. Trastorno mental grave. Adicciones."
            ),
            "condition_on_previous_text": False,
            "no_speech_threshold": 0.6,
            "log_prob_threshold": -1.0,
            "compression_ratio_threshold": 2.2,
        }
        vad_options = {
            "min_silence_duration_ms": 500,
            "speech_pad_ms": 200,
        }

        self.whisper = whisperx.load_model(
            self.cfg.whisper_size, dev,
            compute_type="float16" if dev == "cuda" else "int8",
            language="es",
            asr_options=asr_options,
            vad_method="pyannote",
            vad_options=vad_options,
        )
        self.align, self.align_meta = whisperx.load_align_model("es", dev)
        if self.cfg.do_diarization:
            from whisperx.diarize import DiarizationPipeline
            try:
                self.diar = DiarizationPipeline(device=dev, token=self.cfg.hf_token)
            except TypeError:
                self.diar = DiarizationPipeline(device=dev, use_auth_token=self.cfg.hf_token)
        mem("ASR loaded")

    def _unload_asr(self):
        self.whisper = self.align = self.align_meta = self.diar = None
        clear(); mem("ASR unloaded")

    def _load_emotion(self):
        if self.emotion is not None:
            return
        self.proc, self.emotion = load_emotion(
            "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim", self._dev())
        mem("emotion loaded")

    def _unload_emotion(self):
        self.proc = self.emotion = None
        clear(); mem("emotion unloaded")

    def _load_denoiser(self):
        if self.denoiser is not None: return
        if self.cfg.denoise_method in ("none", "afftdn"): return
        self.denoiser = build_denoiser(
            self.cfg.denoise_method,
            device=self.cfg.denoiser_device,
            fb_model=self.cfg.fb_model,
            fb_wet=self.cfg.fb_wet,
            nr_prop=self.cfg.nr_prop,
        )
        mem("denoiser loaded")

    def _unload_denoiser(self):
        if self.denoiser is None: return
        self.denoiser = None
        clear(); mem("denoiser unloaded")

    # ---- main per-file ----
    def process_file(self, src: str) -> dict:
        t0 = time.time()
        base = os.path.splitext(os.path.basename(src))[0]
        work = os.path.join(self.cfg.temp_dir, base)
        os.makedirs(work, exist_ok=True)
        m = {"file": base, "stages": {}, "paths": {}}
        LOG.info(f"\n=== {base} ===")
        mem("start")

        # 1. Normalize
        norm = normalize_ffmpeg(src, os.path.join(work, "00_norm.wav"))
        m["paths"]["norm"] = norm
        clear(); mem("normalized")

        # 2. Reverb score on pre-denoise signal
        rv = reverb_score_sampled(norm)
        apply_wpe = self.cfg.do_wpe and rv > self.cfg.reverb_threshold
        m["stages"]["reverb"] = {"score": round(rv, 3), "applied": apply_wpe}
        if not self.cfg.do_wpe:
            reason = "WPE disabled by config"
        elif rv <= self.cfg.reverb_threshold:
            reason = f"reverb {rv:.3f} <= threshold {self.cfg.reverb_threshold}"
        else:
            reason = "WPE will run"
        LOG.info(f"[reverb] score={rv:.3f} threshold={self.cfg.reverb_threshold} "
                 f"applied={apply_wpe} ({reason})")

        # 3. WPE (scipy, chunked)
        if apply_wpe:
            wpe_out = wpe_file_chunked(
                norm, os.path.join(work, "01_derev.wav"),
                chunk_s=self.cfg.wpe_chunk_s,
                pad_s=self.cfg.wpe_pad_s,
                xfade_s=self.cfg.wpe_xfade_s,
                taps=self.cfg.wpe_taps,
                delay=self.cfg.wpe_delay,
                iterations=self.cfg.wpe_iters,
                mem_guard_pct=self.cfg.mem_guard_pct,
            )
            if wpe_out is None:
                LOG.warning("[WPE] aborted on memory pressure — using pre-WPE audio")
                derev = norm
                m["stages"]["reverb"]["fallback_no_wpe"] = True
            else:
                derev = wpe_out
            m["paths"]["derev"] = derev
            clear(); mem("wpe done")
        else:
            derev = norm
            m["paths"]["derev"] = norm

        # 4. Denoise → ASR path
        if self.cfg.denoise_method == "none":
            asr_src = derev
            LOG.info("[denoise] skipped")
        else:
            self._load_denoiser()
            LOG.info(f"[denoise] {self.cfg.denoise_method}")
            asr_src = denoise_file(
                derev, os.path.join(work, "02_asr.wav"),
                self.denoiser, self.cfg.denoise_method,
                chunk_s=self.cfg.denoise_chunk_s,
                xfade_s=self.cfg.denoise_xfade_s,
            )
            m["paths"]["asr_src"] = asr_src
            clear(); mem("denoised")

        # 5. Emotion source path
        if not self.cfg.do_emotion:
            emo_src = asr_src
        elif self.cfg.emotion_on_clean_path:
            emo_src = derev
            LOG.info(f"[split] emotion uses CLEAN path: {os.path.basename(derev)}")
        else:
            emo_src = asr_src
            LOG.info(f"[split] emotion uses ASR path: {os.path.basename(asr_src)}")
        m["paths"]["emo_src"] = emo_src
        m["stages"]["split"] = {
            "emotion_on_clean_path": bool(self.cfg.emotion_on_clean_path),
            "same_file": os.path.abspath(asr_src) == os.path.abspath(emo_src),
        }

        # 6. ASR
        self._load_asr()
        LOG.info("[ASR] WhisperX + align + diarize")
        segments = transcribe(asr_src, self.whisper, self.align, self.align_meta,
                              self.diar, self.cfg.min_speakers, self.cfg.max_speakers)
        self._unload_asr()
        m["stages"]["asr"] = {
            "n_segments": len(segments),
            "n_speakers": len(set(s["speaker"] for s in segments)),
            "covered_s": round(sum(s["end"]-s["start"] for s in segments), 1),
        }
        LOG.info(f"[ASR] {len(segments)} segments, "
                 f"{m['stages']['asr']['n_speakers']} speakers")

        # 7. Emotion
        if self.cfg.do_emotion and segments:
            try:
                self._load_emotion()
                LOG.info("[emotion] running on " + os.path.basename(emo_src))
                enriched = emotion_streaming(emo_src, segments, self.proc, self.emotion)
                self._unload_emotion()
                by = {}
                for it in enriched: by.setdefault(it["speaker"], []).append(it)
                for spk, items in by.items():
                    for f in ("valence", "arousal", "dominance"):
                        vals = [x[f] for x in items]
                        mu, sd = float(np.mean(vals)), float(np.std(vals) or 0.01)
                        for x in items:
                            x[f + "_label"] = label3(x[f], mu, sd)
            except Exception as e:
                LOG.exception(f"[emotion] FAILED — continuing without emotion: {e}")
                try:
                    self._unload_emotion()
                except Exception:
                    pass
                enriched = [{**s, "valence": None, "arousal": None, "dominance": None,
                            "valence_label": None, "arousal_label": None,
                            "dominance_label": None} for s in segments]
        else:
            enriched = [{**s, "valence": None, "arousal": None, "dominance": None,
                        "valence_label": None, "arousal_label": None,
                        "dominance_label": None} for s in segments]

        # 8. Write
        srt = os.path.join(self.cfg.output_dir, f"{base}.srt")
        csv = os.path.join(self.cfg.output_dir, f"{base}.emotion.csv")
        js  = os.path.join(self.cfg.output_dir, f"{base}.meta.json")
        write_srt(srt, enriched); write_csv(csv, enriched)
        m["stages"]["total_s"] = round(time.time() - t0, 1)
        with open(js, "w", encoding="utf-8") as f:
            json.dump(m, f, indent=2, ensure_ascii=False)
        LOG.info(f"[✓] {base} → {srt} ({m['stages']['total_s']}s)")

        if not self.cfg.keep_temp:
            shutil.rmtree(work, ignore_errors=True)
        clear(); mem("end file")
        return m

    def run(self):
        for f in os.listdir(self.cfg.input_dir):
            if f.lower().endswith(self.cfg.video_exts):
                s = os.path.join(self.cfg.input_dir, f)
                d = os.path.splitext(s)[0] + ".wav"
                if not os.path.exists(d):
                    LOG.info(f"Extracting audio: {f}")
                    normalize_ffmpeg(s, d)

        files = [os.path.join(self.cfg.input_dir, f)
                 for f in os.listdir(self.cfg.input_dir)
                 if f.lower().endswith(self.cfg.audio_exts)]
        LOG.info(f"{len(files)} files to process")

        summary = []
        try:
            for src in files:
                try:
                    summary.append(self.process_file(src))
                except Exception as e:
                    LOG.exception(f"FAILED {src}: {e}")
                    summary.append({"file": os.path.basename(src), "error": str(e)})
        finally:
            self._unload_denoiser()
            with open(os.path.join(self.cfg.log_dir, "run_summary.json"),
                      "w", encoding="utf-8") as f:
                json.dump(summary, f, indent=2, ensure_ascii=False)
            LOG.info("Done.")


# =========================================================
if __name__ == "__main__":
    from google.colab import drive, userdata
    drive.mount("/content/drive")

    Pipeline(Cfg(
        input_dir="/content/drive/MyDrive/audios_radio",
        output_dir="/content/drive/MyDrive/resultados_radio",
        temp_dir="/content/drive/MyDrive/temporales_audio",
        log_dir="/content/drive/MyDrive/resultados_radio/_logs",
        hf_token=userdata.get("HF_TOKEN"),

        denoise_method="facebook",
        fb_model="master64",
        fb_wet=0.5,
        denoise_chunk_s=30,
        denoise_xfade_s=0.5,
        #nr_prop=0.7,
        do_wpe=True,
        reverb_threshold=0.30,
        wpe_chunk_s=15,
        wpe_pad_s=1,
        wpe_xfade_s=0.3,
        wpe_taps=10,
        wpe_delay=3,
        wpe_iters=3,
        mem_guard_pct=85.0,


        emotion_on_clean_path=True,
        do_emotion=True,
        do_diarization=True,

        whisper_size="large-v3",
        min_speakers=1,
        max_speakers=6,

        keep_temp=False,
    )).run()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
